<a href="https://colab.research.google.com/github/Suhani0411/Graph-Neural-Network-Based-Financial-Fraud-Detection-System/blob/main/GNN_PROJECT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
!pip install torch-geometric

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.4/64.4 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 31.1 MB/s eta 0:00:00


In [5]:
import kagglehub
import pandas as pd
import numpy as np

import torch
import torch.nn as nn
import torch.nn.functional as F

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report
)

from torch_geometric.data import Data
from torch_geometric.nn import (
    GCNConv,
    SAGEConv,
    GATConv
)

In [6]:
# =====================================================
# DOWNLOAD AND LOAD DATASET
# =====================================================

path = kagglehub.dataset_download(
    "ellipticco/elliptic-data-set"
)

print("Dataset Path:", path)

DATA_DIR = f"{path}/elliptic_bitcoin_dataset"

feat_cols = (
    ["txId", "time_step"] +
    [f"f{i}" for i in range(1, 166)]
)

df_features = pd.read_csv(
    f"{DATA_DIR}/elliptic_txs_features.csv",
    header=None,
    names=feat_cols
)

df_edges = pd.read_csv(
    f"{DATA_DIR}/elliptic_txs_edgelist.csv"
)

df_classes = pd.read_csv(
    f"{DATA_DIR}/elliptic_txs_classes.csv"
)

print(df_features.shape)
print(df_edges.shape)
print(df_classes.shape)

100%|██████████| 146M/146M [00:01<00:00, 87.0MB/s]

Extracting files...


Dataset Path: /root/.cache/kagglehub/datasets/ellipticco/elliptic-data-set/versions/1
(203769, 167)
(234355, 2)
(203769, 2)


In [7]:
# =====================================================
# LABEL PROCESSING
# =====================================================

label_map = {
    "1": 1,
    "2": 0,
    "unknown": -1
}

df = df_features.merge(
    df_classes,
    on="txId"
)

df["label"] = (
    df["class"]
    .astype(str)
    .map(label_map)
)

# =====================================================
# FEATURES
# =====================================================

feature_cols = [f"f{i}" for i in range(1,166)]

X = df[feature_cols].values.astype(np.float32)

scaler = StandardScaler()

X = scaler.fit_transform(X)

x = torch.tensor(
    X,
    dtype=torch.float
)

# =====================================================
# LABELS
# =====================================================

y = torch.tensor(
    df["label"].values,
    dtype=torch.long
)

# =====================================================
# NODE MAPPING
# =====================================================

node_map = {
    txid:i
    for i,txid in enumerate(df["txId"])
}

# =====================================================
# EDGE INDEX
# =====================================================

valid_edges = df_edges[
    df_edges["txId1"].isin(node_map)
    &
    df_edges["txId2"].isin(node_map)
]

src = valid_edges["txId1"].map(node_map).values
dst = valid_edges["txId2"].map(node_map).values

edge_index = torch.tensor(
    np.vstack([src,dst]),
    dtype=torch.long
)

# =====================================================
# TIME SPLIT
# =====================================================

time_steps = df["time_step"].values

labels = df["label"].values

train_mask = (
    (labels != -1)
    &
    (time_steps <= 34)
)

val_mask = (
    (labels != -1)
    &
    (time_steps > 34)
    &
    (time_steps <= 42)
)

test_mask = (
    (labels != -1)
    &
    (time_steps > 42)
)

data = Data(
    x=x,
    edge_index=edge_index,
    y=y,
    train_mask=torch.tensor(train_mask),
    val_mask=torch.tensor(val_mask),
    test_mask=torch.tensor(test_mask)
)

print(data)

Data(x=[203769, 165], edge_index=[2, 234355], y=[203769], train_mask=[203769], val_mask=[203769], test_mask=[203769])


In [8]:
class GCN(nn.Module):

    def __init__(self,in_dim,hid_dim,out_dim):
        super().__init__()

        self.conv1 = GCNConv(
            in_dim,
            hid_dim
        )

        self.conv2 = GCNConv(
            hid_dim,
            out_dim
        )

    def forward(self,x,edge_index):

        x = self.conv1(
            x,
            edge_index
        )

        x = F.relu(x)

        x = F.dropout(
            x,
            p=0.5,
            training=self.training
        )

        x = self.conv2(
            x,
            edge_index
        )

        return x


class GraphSAGE(nn.Module):

    def __init__(self,in_dim,hid_dim,out_dim):
        super().__init__()

        self.conv1 = SAGEConv(
            in_dim,
            hid_dim
        )

        self.conv2 = SAGEConv(
            hid_dim,
            out_dim
        )

    def forward(self,x,edge_index):

        x = self.conv1(
            x,
            edge_index
        )

        x = F.relu(x)

        x = F.dropout(
            x,
            p=0.5,
            training=self.training
        )

        x = self.conv2(
            x,
            edge_index
        )

        return x


class GAT(nn.Module):

    def __init__(self,in_dim,hid_dim,out_dim):
        super().__init__()

        self.gat1 = GATConv(
            in_dim,
            hid_dim,
            heads=8
        )

        self.gat2 = GATConv(
            hid_dim*8,
            out_dim,
            heads=1
        )

    def forward(self,x,edge_index):

        x = self.gat1(
            x,
            edge_index
        )

        x = F.elu(x)

        x = F.dropout(
            x,
            p=0.5,
            training=self.training
        )

        x = self.gat2(
            x,
            edge_index
        )

        return x

In [9]:
def train_model(model,data):

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=0.001
    )

    train_labels = data.y[data.train_mask]

    num_licit = (train_labels == 0).sum()
    num_illicit = (train_labels == 1).sum()

    weights = torch.tensor([
      1.0,
      float(num_licit / num_illicit)
  ])

    criterion = nn.CrossEntropyLoss(
      weight=weights
  )
    for epoch in range(100):

        model.train()

        optimizer.zero_grad()

        out = model(
            data.x,
            data.edge_index
        )

        loss = criterion(
            out[data.train_mask],
            data.y[data.train_mask]
        )

        loss.backward()

        optimizer.step()

        if epoch % 10 == 0:
            print(
                f"Epoch {epoch} Loss {loss.item():.4f}"
            )

    return model

In [10]:
def evaluate(model,data,name):

    model.eval()

    with torch.no_grad():

        pred = model(
            data.x,
            data.edge_index
        ).argmax(dim=1)

    y_true = data.y[
        data.test_mask
    ].numpy()

    y_pred = pred[
        data.test_mask
    ].numpy()

    print("\n",name)
    print(
        classification_report(
            y_true,
            y_pred
        )
    )

    return {
        "Model":name,
        "Accuracy":accuracy_score(y_true,y_pred),
        "Precision":precision_score(y_true,y_pred),
        "Recall":recall_score(y_true,y_pred),
        "F1":f1_score(y_true,y_pred)
    }

In [11]:
gcn = GCN(data.num_features,64,2)
gcn = train_model(gcn,data)
gcn_result = evaluate(gcn,data,"GCN")

sage = GraphSAGE(data.num_features,64,2)
sage = train_model(sage,data)
sage_result = evaluate(sage,data,"GraphSAGE")

gat = GAT(data.num_features,8,2)
gat = train_model(gat,data)
gat_result = evaluate(gat,data,"GAT")

results = pd.DataFrame([
    gcn_result,
    sage_result,
    gat_result
])

print(results)
results.to_csv(
    "model_comparison.csv",
    index=False
)

Epoch 0 Loss 1.1163
Epoch 10 Loss 0.5384
Epoch 20 Loss 0.4526
Epoch 30 Loss 0.4158
Epoch 40 Loss 0.3925
Epoch 50 Loss 0.3769
Epoch 60 Loss 0.3601
Epoch 70 Loss 0.3532
Epoch 80 Loss 0.3421
Epoch 90 Loss 0.3234

 GCN
              precision    recall  f1-score   support

           0       0.98      0.74      0.84      6518
           1       0.04      0.41      0.07       169

    accuracy                           0.73      6687
   macro avg       0.51      0.57      0.46      6687
weighted avg       0.96      0.73      0.82      6687

Epoch 0 Loss 0.8888
Epoch 10 Loss 0.4149
Epoch 20 Loss 0.3364
Epoch 30 Loss 0.2953
Epoch 40 Loss 0.2694
Epoch 50 Loss 0.2462
Epoch 60 Loss 0.2337
Epoch 70 Loss 0.2223
Epoch 80 Loss 0.2089
Epoch 90 Loss 0.1973

 GraphSAGE
              precision    recall  f1-score   support

           0       0.98      0.77      0.86      6518
           1       0.04      0.38      0.07       169

    accuracy                           0.76      6687
   macro avg       

In [12]:
import os
print(os.getcwd())

/content


In [13]:
!ls -lh

total 8.0K
-rw-r--r-- 1 root root  288 Jul  3 09:57 model_comparison.csv
drwxr-xr-x 1 root root 4.0K Jun  4 13:32 sample_data


In [14]:
results.to_csv("model_comparison.csv", index=False)
print("CSV saved successfully!")

CSV saved successfully!


In [15]:
from google.colab import files
files.download("model_comparison.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>